# AI-Based Phishing URL Detection System for Banks
## Phase 1 — Dataset Preparation, Baseline Model, and Evaluation

**Task:** Binary classification of `benign` and `phishing` records using engineered URL features from the official ISCX-URL2016 dataset.

**Important scope limitation:** `All.csv` contains precomputed feature values, not raw URL strings. This notebook classifies feature records; it does not accept raw URLs, visit websites, or perform live URL scanning.

### Reproducibility
- Dataset file: `All.csv`
- Target: `URL_Type_obf_Type`
- Selected classes: `benign`, `phishing`
- Split: 80% training / 20% testing, stratified
- Random seed: 42
- Baseline: Logistic Regression (`max_iter=1000`)
- Positive class for phishing metrics: `phishing`

Run the notebook from top to bottom. Upload the original, unmodified `All.csv` when prompted.

## 1. Environment and imports

In [ ]:
import os, sys, json, time, platform, warnings, zipfile
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay, make_scorer
)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

## 2. Upload and load the original dataset

Upload exactly one CSV file: `All.csv`. The notebook does not overwrite or modify the uploaded source file.

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if len(csv_files) != 1:
    raise ValueError(f"Upload exactly one CSV file. Found: {csv_files}")

DATA_PATH = csv_files[0]
if os.path.basename(DATA_PATH).lower() != "all.csv":
    print("WARNING: The uploaded file is not named All.csv. Verify that it is the original ISCX-URL2016 All.csv.")

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

## 3. Original dataset audit

The expected source file contains 36,707 rows and 80 columns: 79 feature columns and one target column. The full dataset contains five classes. The code below reports the actual uploaded file values; do not proceed if the file is not the intended dataset.

In [ ]:
TARGET = "URL_Type_obf_Type"
if TARGET not in df.columns:
    raise KeyError(f"Target column {TARGET!r} was not found.")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Feature columns:", len(df.columns) - 1)
print("\nClass distribution:")
display(df[TARGET].value_counts(dropna=False).rename("count").to_frame())

numeric_df = df.select_dtypes(include=[np.number])
print("Missing cells:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Infinite numeric values:", int(np.isinf(numeric_df.to_numpy()).sum()))
print("\nMissing values by column (non-zero only):")
display(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False).to_frame("missing_count"))

## 4. Define the Phase 1 binary classification problem

Only `benign` and `phishing` records are retained. `Defacement`, `malware`, and `spam` are outside the scope of this binary Phase 1 experiment.

In [ ]:
selected_classes = ["benign", "phishing"]
binary_df = df[df[TARGET].isin(selected_classes)].copy()

print("Binary subset shape before deduplication:", binary_df.shape)
display(binary_df[TARGET].value_counts().rename("count").to_frame())

## 5. Remove exact duplicate records

Exact duplicate records are removed before splitting. This avoids identical complete records being placed in both training and test partitions.

In [ ]:
rows_before = len(binary_df)
duplicates_removed = int(binary_df.duplicated().sum())
binary_df = binary_df.drop_duplicates().reset_index(drop=True)

print("Rows before:", rows_before)
print("Exact duplicate rows removed:", duplicates_removed)
print("Rows after:", len(binary_df))
print("Remaining exact duplicates:", int(binary_df.duplicated().sum()))
print("\nClass distribution after deduplication:")
display(binary_df[TARGET].value_counts().rename("count").to_frame())

## 6. Feature review and reduction

The following six features were excluded based on the prior dataset audit:

1. `ISIpAddressInDomainName`: constant in the selected binary data.
2. `tld`: exact duplicate of `domain_token_count`.
3. `SymbolCount_Domain`: exact linear transformation of `domain_token_count` in this dataset.
4. `subDirLen`: exact duplicate of `pathLength`.
5. `isPortEighty`: near-constant (99.9831% of training values were `-1` in the earlier audit).
6. `executable`: near-constant (99.9916% of training values were `0` in the earlier audit).

The near-constant features were excluded after reviewing their distributions. This is a documented feature-quality decision, not a claim that these features are universally useless.

In [ ]:
reviewed_drop_columns = [
    "ISIpAddressInDomainName", "tld", "SymbolCount_Domain",
    "subDirLen", "isPortEighty", "executable"
]
present_drop_columns = [c for c in reviewed_drop_columns if c in binary_df.columns]
missing_expected_drop_columns = [c for c in reviewed_drop_columns if c not in binary_df.columns]

model_df = binary_df.drop(columns=present_drop_columns).copy()
X = model_df.drop(columns=[TARGET]).copy()
y = model_df[TARGET].copy()

print("Removed columns:", present_drop_columns)
if missing_expected_drop_columns:
    print("Expected columns not found in uploaded file:", missing_expected_drop_columns)
print("Modeling dataset shape:", model_df.shape)
print("Feature count:", X.shape[1])
display(y.value_counts().rename("count").to_frame())

## 7. Missing values, infinite values, and the `-1` sentinel

The audit identified 9 infinite values in `argPathRatio`; these are converted to `NaN`. The original binary modeling dataset had 9,321 missing cells, and converting the 9 infinite values to `NaN` produced 9,330 missing cells.

The meaning of `-1` is not sufficiently documented in the available dataset materials. Therefore, `-1` is preserved as a value and is **not** automatically treated as missing. Missing-value imputation is performed later using training data only.

In [ ]:
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    raise TypeError(f"Unexpected non-numeric feature columns: {non_numeric}")

missing_before_inf = int(X.isna().sum().sum())
inf_before = int(np.isinf(X.to_numpy()).sum())
minus_one_count = int((X == -1).sum().sum())

X = X.replace([np.inf, -np.inf], np.nan)

print("Missing cells before infinity conversion:", missing_before_inf)
print("Infinite values converted:", inf_before)
print("Missing cells after infinity conversion:", int(X.isna().sum().sum()))
print("Remaining infinite values:", int(np.isinf(X.to_numpy()).sum()))
print("Preserved -1 values:", int((X == -1).sum().sum()))

## 8. Stratified train/test split

The data is split into 80% training and 20% testing. Stratification preserves the class proportions. The fixed seed (`42`) supports reproducibility.

The test set is held out and is not used to fit the imputer, scaler, or model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("\nTraining class counts:")
display(y_train.value_counts().rename("count").to_frame())
print("Testing class counts:")
display(y_test.value_counts().rename("count").to_frame())

print("Index overlap:", len(set(X_train.index).intersection(set(X_test.index))))
train_rows = set(map(tuple, X_train.to_numpy()))
test_rows = set(map(tuple, X_test.to_numpy()))
print("Exact feature rows shared across train/test:", len(train_rows.intersection(test_rows)))

## 9. Leakage-safe preprocessing

The preprocessing pipeline performs:
- Median imputation for missing values.
- Addition of missingness-indicator columns.
- Standardization using `StandardScaler`.

The imputer and scaler are fitted on the training partition only, then applied to the test partition. This prevents test-set statistics from influencing preprocessing.

In [ ]:
preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)
print("Training NaN after preprocessing:", int(np.isnan(X_train_processed).sum()))
print("Testing NaN after preprocessing:", int(np.isnan(X_test_processed).sum()))
print("Training Inf after preprocessing:", int(np.isinf(X_train_processed).sum()))
print("Testing Inf after preprocessing:", int(np.isinf(X_test_processed).sum()))

## 10. Train the Logistic Regression baseline

Logistic Regression is the Phase 1 baseline model. The model is trained only on the processed training data. The positive class for phishing-specific metrics is `phishing`.

In [ ]:
baseline_model = LogisticRegression(max_iter=1000, random_state=42)

start_time = time.time()
baseline_model.fit(X_train_processed, y_train)
training_seconds = time.time() - start_time

y_pred = baseline_model.predict(X_test_processed)
phishing_index = list(baseline_model.classes_).index("phishing")
y_proba = baseline_model.predict_proba(X_test_processed)[:, phishing_index]

print(f"Training time: {training_seconds:.4f} seconds")
print("Model classes:", baseline_model.classes_)

## 11. Test-set evaluation

Accuracy measures overall correctness. Precision measures how many samples predicted as phishing are actually phishing. Recall measures how many actual phishing samples are detected. F1-score is the harmonic mean of precision and recall. ROC-AUC and PR-AUC evaluate ranking quality across thresholds.

For a security use case, false negatives (phishing classified as benign) are especially important because they represent missed phishing samples.

In [ ]:
y_test_binary = (y_test == "phishing").astype(int)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision (phishing)": precision_score(y_test, y_pred, pos_label="phishing"),
    "Recall (phishing)": recall_score(y_test, y_pred, pos_label="phishing"),
    "F1-score (phishing)": f1_score(y_test, y_pred, pos_label="phishing"),
    "ROC-AUC": roc_auc_score(y_test_binary, y_proba),
    "PR-AUC": average_precision_score(y_test_binary, y_proba)
}

metrics_df = pd.DataFrame({"Metric": list(metrics.keys()), "Value": list(metrics.values())})
display(metrics_df.style.format({"Value": "{:.4f}"}))

print("\nClassification report:")
print(classification_report(
    y_test, y_pred, labels=["benign", "phishing"], zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=["benign", "phishing"])
print("Confusion matrix (rows=actual; columns=predicted):")
print(cm)

tn, fp, fn, tp = cm.ravel()
print("TN (benign correctly classified):", tn)
print("FP (benign incorrectly flagged as phishing):", fp)
print("FN (phishing missed):", fn)
print("TP (phishing correctly detected):", tp)

### Evaluation plots

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["benign", "phishing"]
).plot(ax=ax, values_format="d", colorbar=False)
plt.title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test_binary, y_proba, ax=ax)
plt.title("Logistic Regression — ROC Curve")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
PrecisionRecallDisplay.from_predictions(y_test_binary, y_proba, ax=ax)
plt.title("Logistic Regression — Precision-Recall Curve")
plt.tight_layout()
plt.show()

## 12. Five-fold stratified cross-validation on the training data

Cross-validation is performed on the training partition only. The entire preprocessing-and-model pipeline is fitted separately inside each fold. This prevents imputation and scaling from being fitted using validation-fold observations.

The reported PR-AUC scorer explicitly selects the probability column for the `phishing` class.

In [ ]:
def phishing_pr_auc(estimator, X_fold, y_fold):
    class_index = list(estimator.classes_).index("phishing")
    probabilities = estimator.predict_proba(X_fold)[:, class_index]
    actual = (np.asarray(y_fold) == "phishing").astype(int)
    return average_precision_score(actual, probabilities)

cv_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label="phishing"),
    "recall": make_scorer(recall_score, pos_label="phishing"),
    "f1": make_scorer(f1_score, pos_label="phishing"),
    "roc_auc": "roc_auc",
    "pr_auc": phishing_pr_auc
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    cv_pipeline, X_train, y_train,
    cv=cv, scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

cv_rows = []
for metric in scoring:
    train_mean = cv_results[f"train_{metric}"].mean()
    valid_mean = cv_results[f"test_{metric}"].mean()
    valid_std = cv_results[f"test_{metric}"].std()
    cv_rows.append({
        "Metric": metric,
        "Train mean": train_mean,
        "Validation mean": valid_mean,
        "Validation std": valid_std,
        "Train-validation gap": train_mean - valid_mean
    })

cv_summary = pd.DataFrame(cv_rows)
display(cv_summary.style.format({
    "Train mean": "{:.4f}", "Validation mean": "{:.4f}",
    "Validation std": "{:.4f}", "Train-validation gap": "{:.4f}"
}))

## 13. Data integrity and leakage-oriented checks

These checks are evidence about the current experiment, not mathematical proof that every possible form of leakage is absent.

- Confirm that the target column is not included in the feature matrix.
- Review feature names for obvious target/label fields.
- Check for duplicate feature columns and perfectly correlated pairs.
- Review the strongest feature-to-target correlations.
- Remember that correlation alone cannot establish or rule out target leakage. Source-level feature-generation review is also required.

In [ ]:
print("Target column:", TARGET)
print("Target in X:", TARGET in X_train.columns)
print("Number of model features:", X_train.shape[1])

suspicious_name_tokens = ["label", "target", "class", "category", "result"]
suspicious_names = [
    col for col in X_train.columns
    if any(token in col.lower() for token in suspicious_name_tokens)
]
print("Feature names containing target-like terms:", suspicious_names)

duplicate_columns = X_train.columns[X_train.T.duplicated()].tolist()
print("Exact duplicate feature columns:", duplicate_columns)

train_corr = X_train.corr(numeric_only=True)
perfect_pairs = []
cols = train_corr.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        value = train_corr.iloc[i, j]
        if pd.notna(value) and abs(value) >= 0.999999:
            perfect_pairs.append((cols[i], cols[j], float(value)))
print("Perfectly correlated feature pairs:", perfect_pairs)

y_train_numeric = (y_train == "phishing").astype(int)
target_corr = X_train.corrwith(y_train_numeric).dropna().sort_values(
    key=lambda s: s.abs(), ascending=False
)
print("\\nTop 20 absolute training-feature/target correlations:")
display(target_corr.head(20).rename("correlation").to_frame())

## 14. Missingness audit and sensitivity check

Missingness itself can sometimes carry predictive information. The following analysis is a diagnostic only. It does not prove leakage. It compares:
1. A model using only missingness indicators.
2. The full feature model with missingness indicators.
3. The full feature model without missingness indicators.

All comparisons use stratified cross-validation within the training partition.

In [ ]:
# Missingness-only experiment: use only binary missing/not-missing indicators.
X_train_missing_flags = X_train.isna().astype(int)

missingness_model = LogisticRegression(max_iter=1000, random_state=42)
missingness_cv = cross_validate(
    missingness_model,
    X_train_missing_flags,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "roc_auc": "roc_auc",
        "f1": make_scorer(f1_score, pos_label="phishing")
    },
    n_jobs=-1
)

print("Missingness-only CV means:")
for metric in ["accuracy", "roc_auc", "f1"]:
    print(
        metric,
        round(missingness_cv[f"test_{metric}"].mean(), 4),
        "+/-",
        round(missingness_cv[f"test_{metric}"].std(), 4)
    )

pipeline_with_indicators = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
pipeline_without_indicators = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=False)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

comparison_scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label="phishing"),
    "recall": make_scorer(recall_score, pos_label="phishing"),
    "f1": make_scorer(f1_score, pos_label="phishing"),
    "roc_auc": "roc_auc",
    "pr_auc": phishing_pr_auc
}
with_ind = cross_validate(
    pipeline_with_indicators, X_train, y_train,
    cv=cv, scoring=comparison_scoring, n_jobs=-1
)
without_ind = cross_validate(
    pipeline_without_indicators, X_train, y_train,
    cv=cv, scoring=comparison_scoring, n_jobs=-1
)

indicator_rows = []
for metric in comparison_scoring:
    with_mean = with_ind[f"test_{metric}"].mean()
    without_mean = without_ind[f"test_{metric}"].mean()
    indicator_rows.append({
        "Metric": metric,
        "With indicators": with_mean,
        "Without indicators": without_mean,
        "Difference": with_mean - without_mean
    })
indicator_comparison = pd.DataFrame(indicator_rows)
display(indicator_comparison.style.format({
    "With indicators": "{:.4f}",
    "Without indicators": "{:.4f}",
    "Difference": "{:+.4f}"
}))

## 15. Interpretation of overfitting and underfitting checks

Compare the training and validation means from the five-fold cross-validation table. In the recorded run, the train-validation gaps were small (approximately 0.0015–0.0020 across the main metrics), which is reassuring and does not show a large train/validation performance gap.

This is not proof that overfitting is impossible. The experiment uses one fixed dataset, one split seed, and one baseline algorithm. Underfitting cannot be ruled out solely from high or low metrics; it should be assessed through model comparison and learning-curve analysis in a later phase.

## 16. Reference results from the recorded local baseline

The previously executed local run produced the following reference values. The Colab outputs should be checked against them. Minor differences may occur if the CSV contents or software versions differ.

| Metric | Reference value |
|---|---:|
| Accuracy | 0.9629 |
| Precision (phishing) | 0.9620 |
| Recall (phishing) | 0.9633 |
| F1-score (phishing) | 0.9627 |
| ROC-AUC | 0.9935 |
| PR-AUC | 0.9940 |

Reference confusion matrix (rows = actual; columns = predicted; order = benign, phishing):

```text
[[1437   56]
 [  54 1418]]
```

The reference run therefore had 54 false negatives and 56 false positives. These are benchmark results on the selected test partition, not production performance guarantees.

## 17. Save reproducibility artifacts

The notebook saves the fitted preprocessing pipeline, trained baseline model, metrics, predictions, confusion matrix, and CV summary. It then packages the output directory into a ZIP file for download.

In [ ]:
OUTPUT_DIR = "phase1_colab_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

import joblib
joblib.dump(preprocessor, os.path.join(OUTPUT_DIR, "preprocessing_pipeline.joblib"))
joblib.dump(baseline_model, os.path.join(OUTPUT_DIR, "logistic_regression_baseline.joblib"))

with open(os.path.join(OUTPUT_DIR, "test_metrics.json"), "w") as f:
    json.dump({k: float(v) for k, v in metrics.items()}, f, indent=4)

pd.DataFrame({
    "actual": y_test.reset_index(drop=True),
    "predicted": pd.Series(y_pred),
    "phishing_probability": y_proba
}).to_csv(os.path.join(OUTPUT_DIR, "test_predictions.csv"), index=False)

pd.DataFrame(
    cm,
    index=["actual_benign", "actual_phishing"],
    columns=["predicted_benign", "predicted_phishing"]
).to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix.csv"))

cv_summary.to_csv(os.path.join(OUTPUT_DIR, "cross_validation_summary.csv"), index=False)
metrics_df.to_csv(os.path.join(OUTPUT_DIR, "test_metrics.csv"), index=False)

print("Saved artifacts:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print("-", filename)

zip_path = "phase1_colab_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for filename in os.listdir(OUTPUT_DIR):
        full_path = os.path.join(OUTPUT_DIR, filename)
        zf.write(full_path, arcname=os.path.join(OUTPUT_DIR, filename))

print("Created:", zip_path)
files.download(zip_path)

## 18. Limitations and Phase 1 conclusion

### Limitations
1. The selected CSV contains engineered features, not raw URL strings.
2. The notebook performs offline classification; it does not visit URLs or query live threat-intelligence feeds.
3. Only benign and phishing classes are included; spam, malware, and defacement are excluded.
4. Only one algorithm (Logistic Regression) is used as the Phase 1 baseline.
5. A random stratified split does not prove generalization to future campaigns, unseen domains, or changing attacker behavior.
6. The meaning of `-1` is not sufficiently documented in the available dataset description; it is preserved rather than assumed to mean missing.
7. Feature correlations and automated checks cannot replace a complete source-level review of how every feature was generated.
8. Results do not establish production readiness or guaranteed protection for banking users.

### Conclusion
The Phase 1 experiment establishes a reproducible binary-classification baseline using the ISCX-URL2016 engineered-feature dataset. The Logistic Regression reference run achieved 96.29% accuracy and 96.27% phishing F1-score on the held-out test set. The next phase should compare multiple algorithms, tune hyperparameters using training-only cross-validation, and investigate explainability, robustness, and more realistic external or temporal evaluation.

### Dataset and research references
1. Canadian Institute for Cybersecurity, **ISCX-URL2016 Dataset**. https://www.unb.ca/cic/datasets/url-2016.html
2. M. S. Islam Mamun, M. A. Rathore, A. H. Lashkari, N. Stakhanova, and A. A. Ghorbani, “Detecting Malicious URLs Using Lexical Analysis,” *Network and System Security (NSS 2016)*, Lecture Notes in Computer Science, vol. 9955, pp. 467–482, 2016. https://doi.org/10.1007/978-3-319-46298-1_30
3. scikit-learn documentation: Pipeline, SimpleImputer, StandardScaler, LogisticRegression, StratifiedKFold, and cross_validate. https://scikit-learn.org/stable/